 " 📊 STB Logs exploration"
 This notebook was created to explore logs from STB ADR APPS
 in regards to find peaks and anomalyes while testing bias scenarios 
   

In [ ]:
#set up environment
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns 
from pathlib import Path 
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline        

#load data
log_file = "../stb/data/raw_logs/7.3.2_v/7_3_2_pe.txt"
#logs_file= "../data/raw_logs/"

In [ ]:
# ============================================================================
# IMPORTS Y SETUP
# ============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import re
from datetime import datetime
from typing import List, Dict, Tuple, Optional
import warnings

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline


# ============================================================================
# FUNCIONES DE PARSING
# ============================================================================

def parse_line(line: str) -> Optional[Dict[str, str]]:
    """
    Parsea una línea de logcat de Android.
    
    Formato esperado:
    MM-DD HH:MM:SS.mmm  PID  TID LEVEL TAG: MESSAGE
    
    Extrae metadata del STB:
    - device_model: B866V2_V1_0_0
    - android_version: Android 12
    - firmware_version: Pendiente (TODO)
    """
    if not line or not line.strip():
        return None
    
    try:
        # Patrón principal de logcat
        pattern = r'^(\d{2}-\d{2})\s+(\d{2}:\d{2}:\d{2}\.\d{3})\s+(\d+)\s+(\d+)\s+([VDIWEF])\s+([^:]+):\s+(.+)$'
        match = re.match(pattern, line)
        
        if not match:
            return None
        
        result = {
            "date": match.group(1),
            "time": match.group(2),
            "pid": match.group(3),
            "tid": match.group(4),
            "level": match.group(5),
            "tag": match.group(6),
            "message": match.group(7)
        }
        
        message = result["message"]
        
        # === REGEX MEJORADO: device_model ===
        # Busca: device_model=B866V2_V1_0_0
        device_match = re.search(
            r'device_model=([A-Z0-9_]+)',
            message,
            re.IGNORECASE
        )
        if device_match:
            result["device_model"] = device_match.group(1)
        
        # === REGEX MEJORADO: android_version ===
        # Busca:
        # - device_so=Android%2012  (URL encoded)
        # - device_so=Android 12
        # - "device_so":"Android 12"
        android_match = re.search(
            r'device_so[=:"]\s*Android(?:%20| )(\d+)',
            message,
            re.IGNORECASE
        )
        if android_match:
            result["android_version"] = android_match.group(1)
        
        # === TODO: firmware_version ===
        # Pendiente - definir qué campo usar (epg_version u otro)
        # Por ahora dejamos None y se marcará como 'Unknown'
        
        return result
        
    except Exception as e:
        print(f"⚠️  Error parseando línea: {e}")
        return None


def time_rounded(time_str: str) -> str:
    """
    Redondea timestamp a minutos (elimina segundos y milisegundos).
    
    Input:  "20:12:37.144"
    Output: "20:12"
    """
    time_obj = datetime.strptime(time_str, "%H:%M:%S.%f")
    return time_obj.replace(second=0, microsecond=0).strftime("%H:%M")


def read_log_file(filepath: str) -> Optional[List[Dict]]:
    """
    Lee archivo de log y retorna lista de logs parseados.
    
    Cada log tiene su timestamp redondeado a minuto.
    """
    parsed_logs = []
    
    try:
        with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
            for i, line in enumerate(f, 1):
                parsed_line = parse_line(line)
                
                if parsed_line:
                    # Redondear tiempo a minuto
                    parsed_line["time"] = time_rounded(parsed_line["time"])
                    parsed_logs.append(parsed_line)
                
                # Progreso cada 10k líneas
                if i % 10000 == 0:
                    print(f"   Procesadas {i:,} líneas...")
        
        print(f"\n✅ Parsing completado: {len(parsed_logs):,} logs válidos de {i:,} líneas totales")
        return parsed_logs
        
    except Exception as e:
        print(f"❌ Error al leer archivo: {e}")
        return None


def group_by_60s(parsed_logs: List[Dict]) -> Dict[str, List[Dict]]:
    """
    Agrupa logs por minuto.
    
    Returns:
        Dict donde key=minuto ("20:12") y value=lista de logs en ese minuto
    """
    logs_by_minute = {}
    
    for log in parsed_logs:
        minute = log['time']
        if minute not in logs_by_minute:
            logs_by_minute[minute] = []
        logs_by_minute[minute].append(log)
    
    return logs_by_minute


def get_claro_pids(logs_by_minute: Dict) -> set:
    """
    Identifica PIDs relacionados con apps de Claro/AMX.
    
    Busca en mensajes: claro, amx, clarotv, launcher
    """
    pattern = r'(?i)(claro|amx\.?|clarotv|launcher)'
    pids_claro = []
    
    for minute, logs in logs_by_minute.items():
        for log in logs:
            if re.search(pattern, log['message']):
                pids_claro.append(log['pid'])
    
    return set(pids_claro)


# ============================================================================
# FUNCIONES DE CONTEO WEF
# ============================================================================

def counter_WEF(logs_by_minute: dict, pids_claro: set) -> dict:
    """
    Cuenta Warnings, Errors y Fatals por minuto.
    
    Diferencia entre:
    - *_all: todos los logs
    - *_claro: solo logs de PIDs de Claro/AMX
    """
    results = {}
    tags_affected = set()
    tags_affected_claro = set()
    
    for minute, logs in logs_by_minute.items():
        w_all, e_all, f_all = 0, 0, 0
        w_claro, e_claro, f_claro = 0, 0, 0
        
        for log in logs:
            level = log['level']
            is_claro = log['pid'] in pids_claro
            
            if level == 'W':
                w_all += 1
                tags_affected.add(log['tag'])
                if is_claro:
                    w_claro += 1
                    tags_affected_claro.add(log['tag'])
            elif level == 'E':
                e_all += 1
                tags_affected.add(log['tag'])
                if is_claro:
                    e_claro += 1
                    tags_affected_claro.add(log['tag'])
            elif level == 'F':
                f_all += 1
                tags_affected.add(log['tag'])
                if is_claro:
                    f_claro += 1
                    tags_affected_claro.add(log['tag'])
        
        results[minute] = {
            'warnings_all': w_all,
            'errors_all': e_all,
            'fatals_all': f_all,
            'warnings_claro': w_claro,
            'errors_claro': e_claro,
            'fatals_claro': f_claro
        }
    
    return results


def print_wef_results(results: dict):
    """
    Imprime los resultados del contador WEF de forma bonita.
    """
    print("\n" + "=" * 80)
    print("📊 RESUMEN DE WARNINGS/ERRORS/FATALS POR MINUTO")
    print("=" * 80)
    print(f"{'Minuto':^8} | {'W_all':>6} {'E_all':>6} {'F_all':>6} | {'W_claro':>8} {'E_claro':>8} {'F_claro':>8}")
    print("-" * 80)
    
    for minute, counts in sorted(results.items()):
        print(f"{minute:^8} | "
              f"{counts['warnings_all']:>6} {counts['errors_all']:>6} {counts['fatals_all']:>6} | "
              f"{counts['warnings_claro']:>8} {counts['errors_claro']:>8} {counts['fatals_claro']:>8}")
    
    print("=" * 80)


# ============================================================================
# FUNCIONES DE TOP ANALYSIS
# ============================================================================

def generate_top_analysis(df: pd.DataFrame, pids_claro: set) -> dict:
    """
    Genera análisis de top 10 para diversos elementos.
    
    Returns:
        Dict con todos los tops
    """
    tops = {}
    
    # Marcar logs de Claro
    df['is_claro'] = df['pid'].isin(pids_claro)
    
    # Top 10 tags más activos (todos)
    tops['top_tags_all'] = df['tag'].value_counts().head(10).to_dict()
    
    # Top 10 tags más activos (solo Claro)
    tops['top_tags_claro'] = df[df['is_claro']]['tag'].value_counts().head(10).to_dict()
    
    # Top 10 mensajes más frecuentes (todos)
    tops['top_messages_all'] = df['message'].value_counts().head(10).to_dict()
    
    # Top 10 mensajes más frecuentes (solo Claro)
    tops['top_messages_claro'] = df[df['is_claro']]['message'].value_counts().head(10).to_dict()
    
    # Top 10 mensajes de WARNING más frecuentes
    tops['top_warnings'] = df[df['level'] == 'W']['message'].value_counts().head(10).to_dict()
    
    # Top 10 mensajes de ERROR más frecuentes
    tops['top_errors'] = df[df['level'] == 'E']['message'].value_counts().head(10).to_dict()
    
    # Top 10 mensajes de FATAL más frecuentes
    tops['top_fatals'] = df[df['level'] == 'F']['message'].value_counts().head(10).to_dict()
    
    # Top 10 PIDs más activos
    tops['top_pids'] = df['pid'].value_counts().head(10).to_dict()
    
    # Top 10 PIDs con más errores
    tops['top_error_pids'] = df[df['level'] == 'E']['pid'].value_counts().head(10).to_dict()
    
    # Top 10 tags con más errores (todos)
    tops['top_error_tags_all'] = df[df['level'] == 'E']['tag'].value_counts().head(10).to_dict()
    
    # Top 10 tags con más errores (solo Claro)
    tops['top_error_tags_claro'] = df[(df['level'] == 'E') & df['is_claro']]['tag'].value_counts().head(10).to_dict()
    
    return tops


def print_top_analysis(tops: dict):
    """
    Imprime de forma bonita el análisis de tops.
    """
    print("\n" + "=" * 70)
    print("🔝 ANÁLISIS DE TOP 10")
    print("=" * 70)
    
    # Top Tags All
    print("\n📊 TOP 10 TAGS MÁS ACTIVOS (TODOS):")
    for i, (tag, count) in enumerate(tops['top_tags_all'].items(), 1):
        print(f"  {i:2d}. {tag:40s} → {count:,}")
    
    # Top Tags Claro
    print("\n📊 TOP 10 TAGS MÁS ACTIVOS (CLARO/AMX):")
    for i, (tag, count) in enumerate(tops['top_tags_claro'].items(), 1):
        print(f"  {i:2d}. {tag:40s} → {count:,}")
    
    # Top Errors
    print("\n❌ TOP 10 MENSAJES DE ERROR:")
    for i, (msg, count) in enumerate(tops['top_errors'].items(), 1):
        short_msg = msg[:60] + "..." if len(msg) > 60 else msg
        print(f"  {i:2d}. [{count:4,}x] {short_msg}")
    
    # Top Warnings
    print("\n⚠️  TOP 10 MENSAJES DE WARNING:")
    for i, (msg, count) in enumerate(tops['top_warnings'].items(), 1):
        short_msg = msg[:60] + "..." if len(msg) > 60 else msg
        print(f"  {i:2d}. [{count:4,}x] {short_msg}")
    
    # Top Fatals
    if tops['top_fatals']:
        print("\n💀 TOP 10 MENSAJES DE FATAL:")
        for i, (msg, count) in enumerate(tops['top_fatals'].items(), 1):
            short_msg = msg[:60] + "..." if len(msg) > 60 else msg
            print(f"  {i:2d}. [{count:4,}x] {short_msg}")
    
    # Top PIDs
    print("\n🎯 TOP 10 PIDs MÁS ACTIVOS:")
    for i, (pid, count) in enumerate(tops['top_pids'].items(), 1):
        print(f"  {i:2d}. PID {pid:6s} → {count:,} logs")
    
    # Top Error PIDs
    print("\n💥 TOP 10 PIDs CON MÁS ERRORES:")
    for i, (pid, count) in enumerate(tops['top_error_pids'].items(), 1):
        print(f"  {i:2d}. PID {pid:6s} → {count:,} errores")
    
    print("\n" + "=" * 70)


# ============================================================================
# FUNCIONES DE VISUALIZACIÓN EDA
# ============================================================================

def plot_eda(df: pd.DataFrame):
    """
    Genera todas las visualizaciones EDA.
    """
    
    # ============ GRÁFICA 1: Distribución de niveles ============
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    
    # Subplot 1: Barras de niveles
    level_counts = df['level'].value_counts()
    axes[0, 0].bar(level_counts.index, level_counts.values, color='steelblue')
    axes[0, 0].set_title('Distribución de Niveles de Log', fontsize=12, fontweight='bold')
    axes[0, 0].set_xlabel('Nivel')
    axes[0, 0].set_ylabel('Cantidad')
    axes[0, 0].grid(axis='y', alpha=0.3)
    
    # Subplot 2: Pie de niveles
    colors = {'V': '#90EE90', 'D': '#87CEEB', 'I': '#FFD700', 
              'W': '#FFA500', 'E': '#FF6347', 'F': '#8B0000'}
    level_colors = [colors.get(level, 'gray') for level in level_counts.index]
    axes[0, 1].pie(level_counts.values, labels=level_counts.index, autopct='%1.1f%%',
                    colors=level_colors, startangle=90)
    axes[0, 1].set_title('Proporción de Niveles', fontsize=12, fontweight='bold')
    
    # Subplot 3: Top 10 Tags
    top_tags = df['tag'].value_counts().head(10)
    axes[1, 0].barh(range(len(top_tags)), top_tags.values, color='teal')
    axes[1, 0].set_yticks(range(len(top_tags)))
    axes[1, 0].set_yticklabels(top_tags.index, fontsize=9)
    axes[1, 0].set_title('Top 10 Tags Más Activos', fontsize=12, fontweight='bold')
    axes[1, 0].set_xlabel('Cantidad')
    axes[1, 0].invert_yaxis()
    axes[1, 0].grid(axis='x', alpha=0.3)
    
    # Subplot 4: Top 10 Tags con errores
    top_error_tags = df[df['level'] == 'E']['tag'].value_counts().head(10)
    axes[1, 1].barh(range(len(top_error_tags)), top_error_tags.values, color='crimson')
    axes[1, 1].set_yticks(range(len(top_error_tags)))
    axes[1, 1].set_yticklabels(top_error_tags.index, fontsize=9)
    axes[1, 1].set_title('Top 10 Tags con Más Errores', fontsize=12, fontweight='bold')
    axes[1, 1].set_xlabel('Cantidad de Errores')
    axes[1, 1].invert_yaxis()
    axes[1, 1].grid(axis='x', alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    
    # ============ GRÁFICA 2: Timeline de W/E/F por minuto ============
    errors_by_min = df[df['level'] == 'E'].groupby('time').size()
    warnings_by_min = df[df['level'] == 'W'].groupby('time').size()
    fatals_by_min = df[df['level'] == 'F'].groupby('time').size()
    
    fig, ax = plt.subplots(figsize=(16, 6))
    
    if len(errors_by_min) > 0:
        ax.plot(errors_by_min.index, errors_by_min.values, 
                label='Errors', color='crimson', linewidth=2, marker='o', markersize=4)
    if len(warnings_by_min) > 0:
        ax.plot(warnings_by_min.index, warnings_by_min.values, 
                label='Warnings', color='orange', linewidth=2, marker='s', markersize=4)
    if len(fatals_by_min) > 0:
        ax.plot(fatals_by_min.index, fatals_by_min.values, 
                label='Fatals', color='darkred', linewidth=2, marker='^', markersize=4)
    
    ax.set_title('Timeline de Warnings/Errors/Fatals por Minuto', fontsize=14, fontweight='bold')
    ax.set_xlabel('Tiempo (minutos)')
    ax.set_ylabel('Cantidad')
    ax.legend()
    ax.grid(alpha=0.3)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
    
    
    # ============ GRÁFICA 3: Comparación Claro vs All ============
    errors_all = df[df['level'] == 'E'].groupby('time').size()
    errors_claro = df[(df['level'] == 'E') & (df['pid'].isin(df[df['is_claro']]['pid'].unique()))].groupby('time').size()
    
    fig, ax = plt.subplots(figsize=(16, 6))
    
    if len(errors_all) > 0:
        ax.plot(errors_all.index, errors_all.values, 
                label='Total Errors', color='steelblue', linewidth=2, alpha=0.7)
    if len(errors_claro) > 0:
        ax.plot(errors_claro.index, errors_claro.values, 
                label='Claro/AMX Errors', color='darkorange', linewidth=2)
    
    ax.set_title('Errores: Total vs Claro/AMX', fontsize=14, fontweight='bold')
    ax.set_xlabel('Tiempo (minutos)')
    ax.set_ylabel('Cantidad de Errores')
    ax.legend()
    ax.grid(alpha=0.3)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
    
    
    # ============ GRÁFICA 4: Heatmap de actividad por minuto ============
    total_by_min = df.groupby('time').size()
    
    fig, ax = plt.subplots(figsize=(16, 4))
    
    ax.bar(total_by_min.index, total_by_min.values, color='mediumpurple', alpha=0.7)
    ax.set_title('Volumen Total de Logs por Minuto', fontsize=14, fontweight='bold')
    ax.set_xlabel('Tiempo (minutos)')
    ax.set_ylabel('Total Logs')
    ax.grid(axis='y', alpha=0.3)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()


# ============================================================================
# EJEMPLO DE USO COMPLETO
# ============================================================================

# Configuración
LOG_FILE = "../../data/raw_logs/unit/log.txt"

# 1. Parsear logs
print(f"📂 Cargando archivo: {LOG_FILE}\n")
parsed_logs = read_log_file(LOG_FILE)

if parsed_logs:
    # 2. Agrupar por minuto
    logs_by_minute = group_by_60s(parsed_logs)
    print(f"📊 Logs agrupados en {len(logs_by_minute)} minutos")
    
    # 3. Identificar PIDs de Claro
    pids_claro = get_claro_pids(logs_by_minute)
    print(f"🎯 PIDs de Claro identificados: {len(pids_claro)}")
    
    # 4. Contador WEF
    results = counter_WEF(logs_by_minute, pids_claro)
    print_wef_results(results)
    
    # 5. Crear DataFrame
    df = pd.DataFrame(parsed_logs)
    for col in ['device_model', 'firmware_version', 'android_version']:
        if col not in df.columns:
            df[col] = None
    
    df['is_error'] = df['level'] == 'E'
    df['is_warning'] = df['level'] == 'W'
    df['is_fatal'] = df['level'] == 'F'
    df['is_claro'] = df['pid'].isin(pids_claro)
    
    # 6. Top Analysis
    tops = generate_top_analysis(df, pids_claro)
    print_top_analysis(tops)
    
    # 7. Visualizaciones
    plot_eda(df)
    
    print("\n✅ Análisis EDA completado!")

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xfc in position 177058: invalid start byte